# Notebook 1 — Guideline Corpus & Benchmark Builder
**Paper:** Can Small Open-Weight Language Models Follow National Dengue Guidelines? An Offline Bilingual Evaluation for Bangladesh and India

**What this notebook does**
1. Gets both guideline PDFs (auto-download, otherwise manual upload)
2. Extracts text with page numbers and builds chunks for RAG
3. Shows Bangladesh vs India pages side by side for a topic (to find conflict items)
4. Creates the benchmark CSV template
5. (Optional) Drafts candidate MCQs with a local LLM, which must then be verified against the guideline page
6. (Optional) Drafts Bangla translations offline with IndicTrans2
7. Validates the benchmark

> All outputs are saved in Google Drive under `/MyDrive/DengueLLM/`.

In [ ]:
# ── 1. Setup ──────────────────────────────────────────────
!pip -q install pymupdf pandas google-genai

from google.colab import drive
drive.mount('/content/drive')

import os, re, json, requests
import fitz  # PyMuPDF
import pandas as pd

BASE = '/content/drive/MyDrive/DengueLLM'
PDF_DIR = f'{BASE}/guidelines'
OUT_DIR = f'{BASE}/data'
os.makedirs(PDF_DIR, exist_ok=True); os.makedirs(OUT_DIR, exist_ok=True)
print('Ready:', BASE)

## 2. Get the guideline PDFs
| Country | Guideline | Source |
|---|---|---|
| Bangladesh | National Guideline for Clinical Management of Dengue, **5th Ed. 2025** (DGHS) | no confirmed public link — upload it if you have it |
| Bangladesh (fallback) | National Guideline for Clinical Management of Dengue Syndrome, 4th Ed. 2018 (revised, printed 2020) | medbox.org |
| India | National Guidelines for Clinical Management of Dengue Fever 2023 (NCVBDC) | ncvbdc.mohfw.gov.in |

**State in the paper which edition was used.**

In [ ]:
# ── 2. Download (fallback: upload) ───────────────────────
SOURCES = {
    'IN_2023': 'https://ncvbdc.mohfw.gov.in/Doc/National%20Guidelines%20for%20Clinical%20Management%20of%20Dengue%20Fever%202023.pdf',
    'BD_2018': 'https://medbox.org/index.php/dl/5e9d82924b9817284241d683',
}

def fetch(key, url):
    path = f'{PDF_DIR}/{key}.pdf'
    if os.path.exists(path) and os.path.getsize(path) > 50_000:
        print(f'✓ {key} already present'); return path
    try:
        r = requests.get(url, timeout=60, headers={'User-Agent': 'Mozilla/5.0'})
        if r.ok and r.content[:4] == b'%PDF':
            open(path, 'wb').write(r.content)
            print(f'✓ {key} downloaded ({len(r.content)//1024} KB)'); return path
        print(f'✗ {key}: not a PDF (status {r.status_code})')
    except Exception as e:
        print(f'✗ {key}: {e}')
    return None

paths = {k: fetch(k, u) for k, u in SOURCES.items()}
print(paths)

In [ ]:
# ── 2b. Manual upload (if you have BD 5th Ed. 2025, or if the download above failed) ──
# After upload, each file is renamed to BD_2025.pdf / BD_2018.pdf / IN_2023.pdf
from google.colab import files
up = files.upload()   # press Cancel to skip
for name, data in up.items():
    key = input(f'Which file is "{name}"? (BD_2025 / BD_2018 / IN_2023): ').strip()
    open(f'{PDF_DIR}/{key}.pdf', 'wb').write(data)
    paths[key] = f'{PDF_DIR}/{key}.pdf'
print(paths)

In [ ]:
# ── 2c. Final selection: use BD 2025 if present, otherwise 2018 ──
avail = {k for k in os.listdir(PDF_DIR)}
BD_KEY = 'BD_2025' if 'BD_2025.pdf' in avail else 'BD_2018'
IN_KEY = 'IN_2023'
USE = {'BD': BD_KEY, 'IN': IN_KEY}
for c, k in USE.items():
    p = f'{PDF_DIR}/{k}.pdf'
    print(c, '→', k, '✓' if os.path.exists(p) else 'MISSING - upload it')

## 3. Text extraction (with page numbers)

In [ ]:
# ── 3. Extract text page-wise ──────────────────────────────
def clean(t):
    t = t.replace('\u00ad', '')
    t = re.sub(r'-\n(?=[a-z])', '', t)        # hyphenated line breaks
    t = re.sub(r'[ \t]+', ' ', t)
    t = re.sub(r'\n{2,}', '\n', t)
    return t.strip()

pages = []
for country, key in USE.items():
    doc = fitz.open(f'{PDF_DIR}/{key}.pdf')
    for i, pg in enumerate(doc, start=1):
        txt = clean(pg.get_text())
        if len(txt) > 80:
            pages.append({'country': country, 'source': key, 'page': i, 'text': txt})
    print(f'{key}: {len(doc)} pages, {sum(p["source"]==key for p in pages)} with text')

pdf_pages = pd.DataFrame(pages)
pdf_pages.to_csv(f'{OUT_DIR}/pages.csv', index=False)
# Scanned PDFs give little text; add an OCR step in that case
pdf_pages.groupby('source')['text'].apply(lambda s: s.str.len().mean()).round()

In [ ]:
# ── 3b. RAG chunks (~300 words, 50 overlap, page metadata kept) ──
def chunk_pages(df, size=300, overlap=50):
    out = []
    for (country, src), g in df.groupby(['country', 'source']):
        words, pmap = [], []
        for _, r in g.iterrows():
            w = r['text'].split(); words += w; pmap += [r['page']] * len(w)
        step = size - overlap
        for s in range(0, max(1, len(words) - overlap), step):
            seg = words[s:s + size]
            if len(seg) < 40: continue
            out.append({'chunk_id': f'{country}_{len(out):04d}', 'country': country, 'source': src,
                        'page_start': pmap[s], 'page_end': pmap[min(s + size, len(words)) - 1],
                        'text': ' '.join(seg)})
    return pd.DataFrame(out)

chunks = chunk_pages(pdf_pages)
chunks.to_json(f'{OUT_DIR}/corpus_chunks.jsonl', orient='records', lines=True, force_ascii=False)
print(chunks.groupby('country').size())

## 4. Bangladesh vs India — topics side by side
Used to find **conflict** items. For each topic the relevant pages of both guidelines are shown; note the page numbers wherever the rules differ.

In [ ]:
# ── 4. Side-by-side topic finder ───────────────────────────
TOPICS = {
    'warning_signs':  ['warning sign'],
    'fluid':          ['crystalloid', 'ml/kg', 'fluid'],
    'platelet_tx':    ['platelet transfusion', 'prophylactic platelet'],
    'diagnosis':      ['NS1', 'IgM', 'tourniquet'],
    'critical_phase': ['critical phase', 'plasma leakage'],
    'shock':          ['shock', 'pulse pressure'],
    'discharge':      ['discharge'],
    'pregnancy':      ['pregnan'],
    'paediatric':     ['infant', 'child'],
    'drugs_avoid':    ['NSAID', 'aspirin', 'steroid'],
}

def find(topic, country, k=3):
    kws = TOPICS[topic]
    d = pdf_pages[pdf_pages.country == country].copy()
    d['hits'] = d.text.str.lower().apply(lambda t: sum(t.count(w.lower()) for w in kws))
    return d[d.hits > 0].sort_values('hits', ascending=False).head(k)

def compare(topic, chars=700):
    for c in ['BD', 'IN']:
        print(f'\n{"="*20} {c} · {topic} {"="*20}')
        for _, r in find(topic, c).iterrows():
            print(f'--- page {r.page} (hits {r.hits}) ---')
            print(r.text[:chars], '...')

compare('platelet_tx')   # change the topic and re-run

In [ ]:
# ── 4b. Topic coverage summary (to plan item counts) ──────
cov = pd.DataFrame({t: {c: len(find(t, c, k=999)) for c in ['BD', 'IN']} for t in TOPICS}).T
cov.columns = ['BD_pages', 'IN_pages']
cov

## 5. Benchmark template
**Target:** BD-only, IN-only and CONFLICT items (same question, different correct answer per country).

For CONFLICT items `answer_BD` and `answer_IN` differ — this is the basis of the **Guideline-Mismatch Rate**.

In [ ]:
# ── 5. Create benchmark template ───────────────────────────
COLS = ['qid', 'set', 'topic', 'question_en', 'A_en', 'B_en', 'C_en', 'D_en',
        'question_bn', 'A_bn', 'B_bn', 'C_bn', 'D_bn',
        'answer_BD', 'answer_IN', 'page_BD', 'page_IN', 'evidence_quote',
        'draft_by', 'verified', 'bn_verified', 'notes']
# set: BD | IN | CONFLICT ; answer_*: A/B/C/D (empty if not applicable)
# draft_by: human | llm ; verified/bn_verified: yes/no

bench_path = f'{OUT_DIR}/benchmark.csv'
if not os.path.exists(bench_path):
    pd.DataFrame(columns=COLS).to_csv(bench_path, index=False)
    print('Template created →', bench_path)
else:
    print('Already exists - not overwritten:', bench_path)

## 6. (Optional) Draft MCQs with a local LLM — Qwen2.5-7B-Instruct, 4-bit, offline
**Needs a GPU runtime.** No API or quota needed. Rules:
- every draft is saved with `draft_by=llm`, `draft_model=qwen2.5-7b`, `verified=no`
- drafts whose evidence quote is not found verbatim on the page are dropped
- every item must be checked against the guideline page before setting `verified=yes`
- the drafting model is flagged separately in the evaluation (NB2 Table S1)

> `data/benchmark.csv` in this repository is already built and checked. Run cells 6–6d only to build a new benchmark.

In [ ]:
# ── 6. Draft MCQs from a page — local Qwen2.5-7B-Instruct (4-bit, offline, no API) ──
import subprocess, sys, gc, torch, transformers
subprocess.run([sys.executable, '-m', 'pip', '-q', 'install', 'bitsandbytes>=0.43'], check=True)
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
assert torch.cuda.is_available(), 'GPU required - Runtime -> Change runtime type -> T4 GPU'

DRAFT_MODEL, DRAFT_ID = 'qwen2.5-7b', 'Qwen/Qwen2.5-7B-Instruct'

# tag earlier Gemini drafts with draft_model='gemini-2.5-flash' (runs once)
def tag_draft_model(name=DRAFT_MODEL):
    b = pd.read_csv(bench_path, dtype=str).fillna('')
    if 'draft_model' not in b: b['draft_model'] = ''
    m = (b.draft_by == 'llm') & (b.draft_model == '')
    b.loc[m, 'draft_model'] = name; b.to_csv(bench_path, index=False)
    print(f'draft_model={name} → {m.sum()} rows |', b[b.draft_by == 'llm'].draft_model.value_counts().to_dict())
if not os.path.exists(f'{OUT_DIR}/gemini_tagged.flag'):
    tag_draft_model('gemini-2.5-flash'); open(f'{OUT_DIR}/gemini_tagged.flag', 'w').write('done')

cdtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
dkey = 'dtype' if tuple(int(x) for x in re.findall(r'\d+', transformers.__version__)[:2]) >= (4, 56) else 'torch_dtype'
_tok = AutoTokenizer.from_pretrained(DRAFT_ID)
_llm = AutoModelForCausalLM.from_pretrained(DRAFT_ID, device_map={'': 0}, **{dkey: cdtype},
        quantization_config=BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4',
                                               bnb_4bit_compute_dtype=cdtype, bnb_4bit_use_double_quant=True)).eval()
print('Loaded', DRAFT_ID, '| VRAM', round(torch.cuda.memory_allocated() / 1e9, 1), 'GB')

def extract_json(s):
    s = re.sub(r'```(?:json)?', '', s)
    m = re.search(r'\[.*\]', s, re.S)
    try: items = json.loads(m.group(0)) if m else []
    except Exception: items = []
    need = ['question', 'A', 'B', 'C', 'D']
    return [it for it in items if isinstance(it, dict) and all(str(it.get(k, '')).strip() for k in need)]

@torch.inference_mode()
def local_generate(prompt, max_new_tokens=1500):
    msgs = [{'role': 'user', 'content': prompt + '\n\nReturn ONLY the JSON list, no other text.'}]
    enc = _tok(_tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True), return_tensors='pt').to(0)
    out = _llm.generate(**enc, max_new_tokens=max_new_tokens, do_sample=False, temperature=None, top_p=None, top_k=None)
    return _tok.decode(out[0, enc['input_ids'].shape[1]:], skip_special_tokens=True)

# same interface as the Gemini client, so cells 6b and 6c work unchanged
class _Resp:
    def __init__(self, t): self.text = t
class _Models:
    def generate_content(self, model=None, contents='', config=None):
        return _Resp(json.dumps(extract_json(local_generate(contents)), ensure_ascii=False))
class _LocalClient:
    models = _Models()
client = _LocalClient()

PROMPT = '''You are helping build a benchmark from an official dengue clinical guideline.
From the guideline text below, write {n} multiple-choice questions that test clinically
important, checkable rules (thresholds, doses, criteria, do/don't). Use ONLY facts stated in
the text. Each question has 4 options (A-D), exactly one correct.
Return a JSON list of objects with keys: question, A, B, C, D, answer, evidence_quote
(evidence_quote = exact sentence copied from the text).

GUIDELINE TEXT ({country}, page {page}):
<<<{text}>>>'''

def draft_from_page(country, page, n=3):
    row = pdf_pages[(pdf_pages.country == country) & (pdf_pages.page == page)].iloc[0]
    resp = client.models.generate_content(model=DRAFT_MODEL,
             contents=PROMPT.format(n=n, country=country, page=page, text=row.text[:6000]))
    items = [it for it in json.loads(resp.text) if str(it.get('answer', '')).strip().upper()[:1] in list('ABCD')]
    norm = lambda s: re.sub(r'\s+', ' ', str(s).lower()).strip()
    for it in items:
        it['answer'] = str(it['answer']).strip().upper()[:1]; it.setdefault('evidence_quote', '')
        it['quote_found'] = norm(it.get('evidence_quote', ''))[:60] in norm(row.text)
    return items

def free_drafter():
    global _llm
    del _llm; gc.collect(); torch.cuda.empty_cache(); print('Drafter removed from GPU')

items = draft_from_page('BD', page=int(find('platelet_tx', 'BD').iloc[0].page))
pd.DataFrame(items)

In [ ]:
# ── 6b. Safe qid numbering + answer-letter balancing ──
import shutil, numpy as np, time

def next_qid(b):
    n = b['qid'].str.extract(r'Q(\d+)')[0].dropna().astype(int)
    return (n.max() if len(n) else 0) + 1

def add_drafts(items, country, page, topic, set_name=None):
    b = pd.read_csv(bench_path, dtype=str).fillna('')
    n0, rows = next_qid(b), []
    for it in items:
        if not it.get('quote_found'):
            print('skip (quote not found on page):', it['question'][:60]); continue
        rows.append({'qid': f'Q{n0 + len(rows):03d}', 'set': set_name or country, 'topic': topic,
                     'question_en': it['question'], 'A_en': it['A'], 'B_en': it['B'], 'C_en': it['C'], 'D_en': it['D'],
                     f'answer_{country}': it['answer'], f'page_{country}': page,
                     'evidence_quote': it['evidence_quote'], 'draft_by': 'llm', 'draft_model': globals().get('DRAFT_MODEL', ''),
                     'verified': 'no', 'bn_verified': 'no'})
    b = pd.concat([b, pd.DataFrame(rows)], ignore_index=True)
    b.to_csv(bench_path, index=False); print(f'+{len(rows)} rows → total {len(b)}')

SKIP = re.compile(r'above|both|none of|all of|উপরের|উভয়|কোনোটিই|সবগুলো', re.I)
def balance_answers(save=False, seed=42):
    # applies the same A-D permutation to EN and BN options and remaps answer_BD/answer_IN
    b = pd.read_csv(bench_path, dtype=str).fillna(''); L = list('ABCD')
    print('before:', pd.concat([b.answer_BD, b.answer_IN]).replace('', np.nan).dropna().value_counts().to_dict())
    rng = np.random.default_rng(seed)
    for i, r in b.iterrows():
        opts = [r[f'{x}_{l}'] for l in ['en', 'bn'] for x in L]
        if any(SKIP.search(o) for o in opts) or not all(r[f'{x}_en'] for x in L): continue
        perm = rng.permutation(4)                      # new position j <- old option perm[j]
        for l in ['en', 'bn']:
            old = [r[f'{x}_{l}'] for x in L]
            for j, x in enumerate(L): b.at[i, f'{x}_{l}'] = old[perm[j]]
        new_of = {L[perm[j]]: L[j] for j in range(4)}
        for c in ['answer_BD', 'answer_IN']:
            if r[c] in new_of: b.at[i, c] = new_of[r[c]]
    print('after:', pd.concat([b.answer_BD, b.answer_IN]).replace('', np.nan).dropna().value_counts().to_dict())
    if save:
        shutil.copy(bench_path, bench_path.replace('.csv', f'_backup_{time.strftime("%Y%m%d_%H%M")}.csv'))
        b.to_csv(bench_path, index=False); print('saved (backup kept)')

# once all items and Bangla versions exist: preview with balance_answers(), then balance_answers(save=True)

In [ ]:
# ── 6c. Bulk draft: BD-only, IN-only, CONFLICT (resume-safe) ──
import time
def next_qid(b):
    n = b['qid'].str.extract(r'Q(\d+)')[0].dropna().astype(int)
    return (n.max() if len(n) else 0) + 1

norm = lambda s: re.sub(r'\s+', ' ', str(s).lower()).strip()
LOG = f'{OUT_DIR}/draft_log.json'
log = json.load(open(LOG)) if os.path.exists(LOG) else []
save_log = lambda: json.dump(log, open(LOG, 'w'))

def retry(fn, *a):
    for t in range(3):
        try: return fn(*a)
        except Exception as e: print('  retry:', str(e)[:100]); time.sleep(5)
    return []

CPROMPT = '''Below are excerpts on the same topic from two national dengue clinical guidelines.
Find rules where the two guidelines give DIFFERENT recommendations (threshold, dose, criterion, timing, do/don't).
For each, write one multiple-choice question (options A-D) that can be asked identically for both countries,
where the correct answer under the Bangladesh guideline differs from that under the India guideline.
Use ONLY facts stated in the excerpts. If there is no real difference, return [].
Return a JSON list of objects with keys: question, A, B, C, D, answer_BD, answer_IN, quote_BD, quote_IN
(quotes = exact sentences copied from each excerpt).

BANGLADESH (page {pb}):
<<<{tb}>>>

INDIA (page {pi}):
<<<{ti}>>>'''

def draft_conflict(rb, ri):
    resp = client.models.generate_content(model=DRAFT_MODEL,
        contents=CPROMPT.format(pb=rb.page, tb=rb.text[:5000], pi=ri.page, ti=ri.text[:5000]))
    return json.loads(resp.text)

def add_conflicts(items, rb, ri, topic):
    b = pd.read_csv(bench_path, dtype=str).fillna(''); n0, rows = next_qid(b), []
    for it in items:
        ok = (norm(it.get('quote_BD', ''))[:60] in norm(rb.text) and norm(it.get('quote_IN', ''))[:60] in norm(ri.text)
              and it.get('answer_BD') in list('ABCD') and it.get('answer_IN') in list('ABCD')
              and it['answer_BD'] != it['answer_IN'])
        if not ok: print('  skip conflict:', it.get('question', '')[:60]); continue
        rows.append({'qid': f'Q{n0 + len(rows):03d}', 'set': 'CONFLICT', 'topic': topic,
                     'question_en': it['question'], 'A_en': it['A'], 'B_en': it['B'], 'C_en': it['C'], 'D_en': it['D'],
                     'answer_BD': it['answer_BD'], 'answer_IN': it['answer_IN'], 'page_BD': rb.page, 'page_IN': ri.page,
                     'evidence_quote': f"BD: {it['quote_BD']} || IN: {it['quote_IN']}",
                     'draft_by': 'llm', 'draft_model': DRAFT_MODEL, 'verified': 'no', 'bn_verified': 'no'})
    if rows:
        pd.concat([b, pd.DataFrame(rows)], ignore_index=True).to_csv(bench_path, index=False)
        print(f'+{len(rows)} CONFLICT ({topic})')

# (a) BD-only and IN-only: 2 pages per topic per country x 3 items
for topic in TOPICS:
    for c in ['BD', 'IN']:
        for pg in find(topic, c, k=2).page.tolist():
            tag = f'{c}|{pg}'
            if tag in log: continue
            items = retry(draft_from_page, c, int(pg), 3)
            if items: add_drafts(items, c, int(pg), topic)
            log.append(tag); save_log()
# (b) CONFLICT: same-topic pages of both countries side by side
for topic in TOPICS:
    for rb, ri in zip(find(topic, 'BD', k=2).itertuples(), find(topic, 'IN', k=2).itertuples()):
        tag = f'CONF|{rb.page}|{ri.page}'
        if tag in log: continue
        items = retry(draft_conflict, rb, ri)
        if items: add_conflicts(items, rb, ri, topic)
        log.append(tag); save_log()
print('\nAll pages done')
if 'tag_draft_model' in globals(): tag_draft_model()
b = pd.read_csv(bench_path, dtype=str).fillna('')
print('Drafts in benchmark:', b.set.value_counts().to_dict())

## 6d. (Optional) Bangla draft translation — IndicTrans2 (en→bn), offline
IndicTrans2's model code does not run on the newest transformers, so this cell runs it in a separate environment with transformers 4.51.3.
`ai4bharat/indictrans2-en-indic-1B` is gated: request access on Hugging Face and store `HF_TOKEN` in Secrets.

In [ ]:
# ── 6d. Bangla draft translation — IndicTrans2 (en->bn, offline); verify and set bn_verified=yes afterwards ──
# IndicTrans2's model code fails on new transformers -> run it in a separate env with transformers 4.51.3
import subprocess, sys, os, gc, json, hashlib
ONLY_VERIFIED = True          # True = translate only items whose English is verified
IT_ENV = '/content/it_env'
IT_OUT = f'{OUT_DIR}/indictrans_out.jsonl'          # kept in Drive so an interrupted run resumes
if 'tag_draft_model' in globals(): tag_draft_model()
if '_llm' in globals(): free_drafter()

if not os.path.exists(f'{IT_ENV}/transformers/__init__.py'):
    print('Building separate environment (first run, ~1 minute)...')
    subprocess.run([sys.executable, '-m', 'pip', '-q', 'install', '--target', IT_ENV, '--no-deps',
                    'transformers==4.51.3', 'tokenizers==0.21.1', 'huggingface_hub==0.34.4', 'indictranstoolkit'], check=True)
    subprocess.run([sys.executable, '-m', 'pip', '-q', 'install', 'indic-nlp-library', 'sacremoses', 'sentencepiece'], check=True)

JOB = r'''
import sys, os, json, torch
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
from IndicTransToolkit import IndicProcessor
ID, SRC, TGT = "ai4bharat/indictrans2-en-indic-1B", "eng_Latn", "ben_Beng"
inp, outp = sys.argv[1], sys.argv[2]
data = json.load(open(inp, encoding="utf-8"))
done = set()
if os.path.exists(outp):
    done = {json.loads(l)["key"] for l in open(outp, encoding="utf-8") if l.strip()}
todo = [d for d in data if d["key"] not in done]
print("translate:", len(todo), flush=True)
if not todo: sys.exit(0)
hf = os.environ.get("HF_TOKEN")
tok = AutoTokenizer.from_pretrained(ID, trust_remote_code=True, token=hf)
model = AutoModelForSeq2SeqLM.from_pretrained(ID, trust_remote_code=True, token=hf, torch_dtype=torch.float16).to("cuda").eval()
ip = IndicProcessor(inference=True)
use_cache = True
with open(outp, "a", encoding="utf-8") as fo:
    for i in range(0, len(todo), 16):
        ch = todo[i:i + 16]
        batch = ip.preprocess_batch([d["text"] for d in ch], src_lang=SRC, tgt_lang=TGT)
        enc = tok(batch, truncation=True, padding="longest", return_tensors="pt", return_attention_mask=True).to("cuda")
        kw = dict(min_length=0, max_length=256, num_beams=5, num_return_sequences=1)
        with torch.inference_mode():
            try:
                gen = model.generate(**enc, use_cache=use_cache, **kw)
            except Exception as e:
                if not use_cache: raise
                print("use_cache=False:", str(e)[:80], flush=True); use_cache = False
                gen = model.generate(**enc, use_cache=False, **kw)
        dec = tok.batch_decode(gen, skip_special_tokens=True, clean_up_tokenization_spaces=True)
        for d, t in zip(ch, ip.postprocess_batch(dec, lang=TGT)):
            fo.write(json.dumps({"key": d["key"], "bn": t}, ensure_ascii=False) + "\n")
        fo.flush()
        print(f"  {min(i + 16, len(todo))}/{len(todo)}", flush=True)
'''
open('/content/it_job.py', 'w').write(JOB)

# ── items to translate
b = pd.read_csv(bench_path, dtype=str).fillna('')
for c in ['bn_draft_by', 'notes']:
    if c not in b: b[c] = ''
F = ['question', 'A', 'B', 'C', 'D']
key = lambda q, f, t: f'{q}|{f}|{hashlib.md5(t.encode()).hexdigest()[:8]}'
rows = b[(b.question_en != '') & (b.question_bn == '') & ((b.verified == 'yes') | (not ONLY_VERIFIED))]
print('Items to translate:', len(rows))
items = [{'key': key(r.qid, f, r[f'{f}_en']), 'text': r[f'{f}_en']} for _, r in rows.iterrows() for f in F]
json.dump(items, open('/content/it_in.json', 'w', encoding='utf-8'), ensure_ascii=False)

# ── run translation in a separate process (streams its output)
if items:
    env = dict(os.environ, PYTHONPATH=IT_ENV + ':' + os.environ.get('PYTHONPATH', ''),
               HF_TOKEN=userdata.get('HF_TOKEN'), TOKENIZERS_PARALLELISM='false')
    p = subprocess.Popen([sys.executable, '/content/it_job.py', '/content/it_in.json', IT_OUT],
                         env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in p.stdout:
        if 'Warning' not in line and 'warn(' not in line: print(line, end='')
    p.wait()
    if p.returncode != 0: print(f'WARNING: translation process stopped (code {p.returncode}); finished items are saved below')

# ── write results (only when all 5 fields are translated) and check that numbers match
BN_DIG = str.maketrans('০১২৩৪৫৬৭৮৯', '0123456789')
nums = lambda s: set(re.findall(r'\d+(?:\.\d+)?', s.translate(BN_DIG)))
tr = {}
if os.path.exists(IT_OUT):
    for l in open(IT_OUT, encoding='utf-8'):
        if l.strip(): d = json.loads(l); tr[d['key']] = d['bn']
n_done = 0
for i, r in rows.iterrows():
    ks = [key(r.qid, f, r[f'{f}_en']) for f in F]
    if not all(k in tr for k in ks): continue
    bad = [f for f, k in zip(F, ks) if not nums(r[f'{f}_en']) <= nums(tr[k])]
    for f, k in zip(F, ks): b.at[i, f'{f}_bn'] = tr[k]
    b.at[i, 'bn_verified'] = 'no'; b.at[i, 'bn_draft_by'] = 'indictrans2-en-indic-1B'
    if bad: b.at[i, 'notes'] = (b.at[i, 'notes'] + f' | CHECK numbers in BN: {",".join(bad)}').strip(' |')
    n_done += 1
b.to_csv(bench_path, index=False)
flag = b.notes.str.contains('CHECK numbers', na=False).sum()
print(f'Saved Bangla drafts for {n_done} items | number mismatches: {flag} (see "CHECK numbers" in notes)')
b[b.bn_draft_by != ''][['qid', 'question_en', 'question_bn']].head(3)

## 7. Validation check
Run this cell after editing the benchmark CSV (for example in Google Sheets).

In [ ]:
# ── 7. Validate benchmark ──────────────────────────────────
b = pd.read_csv(bench_path, dtype=str).fillna('')
issues = []
for _, r in b.iterrows():
    q = r.qid
    if r.set not in {'BD', 'IN', 'CONFLICT'}: issues.append((q, 'invalid set'))
    need = ['answer_BD'] if r.set == 'BD' else ['answer_IN'] if r.set == 'IN' else ['answer_BD', 'answer_IN']
    for col in need:
        if r[col] not in list('ABCD'): issues.append((q, f'{col} empty or invalid'))
    if r.set == 'CONFLICT' and r.answer_BD == r.answer_IN: issues.append((q, 'CONFLICT but both answers equal'))
    if any(not r[c] for c in ['question_en', 'A_en', 'B_en', 'C_en', 'D_en']): issues.append((q, 'English incomplete'))
    if any(not r[c] for c in ['question_bn', 'A_bn', 'B_bn', 'C_bn', 'D_bn']): issues.append((q, 'Bangla missing'))
    if r.verified != 'yes': issues.append((q, 'not verified'))
    if r.bn_verified != 'yes': issues.append((q, 'Bangla not verified'))
dups = b[b.duplicated('question_en', keep=False)].qid.tolist()
issues += [(q, 'duplicate question') for q in dups]

print(f'Total: {len(b)} | BD {sum(b.set=="BD")} · IN {sum(b.set=="IN")} · CONFLICT {sum(b.set=="CONFLICT")}')
print(f'Verified: {sum(b.verified=="yes")} | Bangla verified: {sum(b.bn_verified=="yes")}')
pd.DataFrame(issues, columns=['qid', 'issue']).groupby('issue').size() if issues else print('All checks passed')